# Bouwen van een Telecom-Omzetbewaking Samenvattingskubus met PROC SUMMARY

## Samenvatting

Een omzetbewakingsteam (revenue assurance) bij een mobiele aanbieder aggregeert vooraf een maand aan abonnee-dagfactuurgegevens tot een compacte samenvattingskubus, zodat analisten kunnen inzoomen op verrekende omzet per regio, abonnementsniveau en gesprekstype zonder de ruwe feitentabel opnieuw te doorzoeken. `PROC SUMMARY` rolt 100 abonnee-dagrecords op tot een multidimensionale verzameling cellen: het fijnste niveau (regio x abonnementsniveau x gesprekstype) vult 25 van de 27 mogelijke cellen, terwijl benoemde subkubussen de marginalen leveren waar analisten het vaakst naar vragen. In deze steekproefmaand verrekende de aanbieder $222,78 over de drie actieve regio's, waarbij Zuid ($97,44) en Oost ($86,94) samen 83% van de omzet uitmaken en Noord achterblijft op $38,40. De rijkste afzonderlijke subkubus is de Plus-niveau Spraakdienst ($59,06 over 18 records), en het rangschikken van cellen op opbrengst per record laat datagebruikcellen naar voren komen als de meest waardevolle doelen voor een lekkage-audit (hoogste opbrengst $7,87/record). Elk cijfer hieronder is rechtstreeks afgelezen uit de uitgevoerde output.

## Gegevensbronnen

| Dataset | Rijen | Detailniveau | Kernvariabelen |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Eén rij per abonnee-dag gebruikssamenvatting | `region` (Oost/Zuid/Noord), `plan_tier` (Prepaid/Basis/Plus), `call_type` (Spraak/SMS/Data), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Eén rij per niet-lege (region x plan_tier x call_type) cel | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Eén rij per cel van de benoemde drill-subkubussen | `_TYPE_`, `_FREQ_`, `rev_total` |

Alle data wordt inline gegenereerd met `call streaminit()` / `rand()`; er zijn geen externe bestanden of netwerktoegang nodig. Deze omgeving draait ongelicentieerd, dus geschreven datasets zijn beperkt tot 100 observaties - het scenario is zo gedimensioneerd dat de kubus binnen die limiet volledig gevuld is.

## Van ruwe call-detail records naar een doorzoekbare kubus

Mobiele aanbieders verrekenen omzet over miljoenen dagelijkse gebruiksgebeurtenissen. Om omzetbewakingsanalisten vragen te laten beantwoorden zoals *"Wat was de Plus-niveau spraakomzet in de regio Zuid vorige maand?"* zonder de ruwe feitentabel elke keer opnieuw te doorzoeken, **aggregeren we de data vooraf** tot een compacte samenvattingskubus.

`PROC SUMMARY` is het werkpaard van Base SAS voor precies dit doel: het groepeert een platte feitentabel op een of meer `CLASS`-dimensies en schrijft de gevraagde statistieken naar een outputdataset, waarbij elke rij wordt gemarkeerd met `_TYPE_` (welke dimensies actief zijn) en `_FREQ_` (records achter de cel). Die outputdataset *is* een multidimensionale kubus - dezelfde roll-upstructuur die een OLAP-tool zou tonen, gematerialiseerd als een gewone SAS-dataset die je kunt afdrukken, joinen en doorsnijden.

Dit notebook:

1. Genereert een realistische maand aan abonnee-dagfactuurgegevens.
2. Bouwt de fijnste-korrel kubus (regio x abonnementsniveau x gesprekstype) met `PROC SUMMARY NWAY`.
3. Materialiseert benoemde **drill-subkubussen** met de `TYPES`-instructie.
4. Projecteert omzet naar de abonneebasis met een `WEIGHT`, zoomt in op één regio, en rangschikt cellen op opbrengst per record voor lekkage-triage.

## Stap 1 - Genereer synthetische call-detail-/factuurgegevens

Elke rij vat het gebruik van één abonnee van één dienst op één dag samen. We gebruiken `call streaminit` voor reproduceerbaarheid en `rand()` om plausibele verdelingen te trekken: omzet en gebruik schalen met het abonnementsniveau, spraakomzet volgt de factureerbare minuten, en dataomzet volgt de megabytes. Elke `RAND('table', ...)` krijgt één kans per categorie, zodat elke regio, elk niveau en elk gesprekstype voorkomt in de steekproef van 100 records. Een klein `subscriber_wt` steekproefgewicht wordt toegevoegd zodat we later een gewogen maatstaf kunnen demonstreren.

In [1]:
GEGEVENS work.cdr_billing;
    CALL streaminit(20260131);
    LENGTE region $6 plan_tier $9 call_type $7 device_class $14 bill_month $7;
    label revenue       = "Verrekende Omzet (USD)"
          call_minutes  = "Factureerbare Spraakminuten"
          data_mb       = "Datavolume (MB)"
          subscriber_wt = "Abonnee-steekproefgewicht";

    DOE i = 1 TOT 100;
        /* --- Dimensies: een kans per niveau, optellend tot 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        SELECTEREN (r);
            WANNEER (1) region = "Oost";
            WANNEER (2) region = "Zuid";
            ANDERS_WEL region = "Noord";
        EINDE;

        p = rand("table", 0.30, 0.40, 0.30);
        SELECTEREN (p);
            WANNEER (1) plan_tier = "Prepaid";
            WANNEER (2) plan_tier = "Basis";
            ANDERS_WEL plan_tier = "Plus";
        EINDE;

        c = rand("table", 0.50, 0.30, 0.20);
        SELECTEREN (c);
            WANNEER (1) call_type = "Spraak";
            WANNEER (2) call_type = "SMS";
            ANDERS_WEL call_type = "Data";
        EINDE;

        ALS rand("uniform") < 0.55 DAN device_class = "Smart";
        ANDERS device_class = "Instaptoestel";

        bill_month = "2026-01";

        /* --- Maatstaven, geschaald per niveau en dienst --- */
        tier_mult = (plan_tier = "Prepaid")*0.7
                  + (plan_tier = "Basis")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Spraak")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Data")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        UITVOER;
    EINDE;
    VERWIJDEREN i r p c tier_mult base_rev;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=work.cdr_billing(obs=8) label noobs;
    TITEL "Voorbeeld Abonnee-Dagfactuurgegevens";
UITVOEREN;


                                          Voorbeeld Abonnee-Dagfactuurgegevens                                          

region  plan_tier  call_type   device_class  bill_month  Factureerbare Spraakminuten  Datavolume (MB)  Verrekende Omzet (USD)  Abonnee-steekproefgewicht
Noord   Plus       SMS        Smart          2026-01                               0                0                    0.67                       1.13
Zuid    Prepaid    SMS        Instaptoestel  2026-01                               0                0                    0.94                       1.42
Zuid    Plus       SMS        Smart          2026-01                               0                0                    0.99                       0.86
Zuid    Plus       SMS        Smart          2026-01                               0                0                    1.01                       1.03
Zuid    Plus       Spraak     Smart          2026-01                           103.4                0            


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Stap 2 - Bouw de fijnste-korrel kubus met PROC SUMMARY NWAY

`NWAY` behoudt alleen de meest gedetailleerde combinatie van alle `CLASS`-dimensies - hier elke gevulde (region x plan_tier x call_type) cel. Voor elke cel bewaren we de omzet `SUM`, `MEAN` en `MAX` plus de totale minuten en megabytes, zodat een analist de totale omzet per cel kan aflezen, het gemiddelde per record kan afleiden, en de grootste afzonderlijke kosten kan opsporen. `_FREQ_` registreert hoeveel abonnee-dagen achter elke cel zitten. We laten `_TYPE_` hier weg omdat, op het NWAY-niveau, elke rij hetzelfde type heeft.

In [2]:
PROCEDURE summary GEGEVENS=work.cdr_billing NWAY;
    KLASSE region plan_tier call_type;
    VARIABELE revenue call_minutes data_mb;
    UITVOER out=work.cube_nway(VERWIJDEREN=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=work.cube_nway(obs=12) noobs label;
    label region='Regio' plan_tier='Abonnementsniveau' call_type='Gesprekstype'
          rev_total='Totale Omzet' rev_mean='Gemiddelde Omzet' rev_max='Max Omzet'
          min_total='Totaal Belminuten' data_total='Totaal Data (MB)';
    TITEL "NWAY-kubuscellen (regio * abonnementsniveau * gesprekstype)";
    OPMAAK rev_total rev_mean rev_max min_total data_total comma10.2;
UITVOEREN;


                              NWAY-kubuscellen (regio * abonnementsniveau * gesprekstype)                               

Regio  Abonnementsniveau  Gesprekstype  _FREQ_  Totale Omzet  Gemiddelde Omzet  Max Omzet  Totaal Belminuten  Totaal Data (MB)
Noord  Basis              Data               1          7.87              7.87       7.87               0.00            725.10
Noord  Basis              SMS                2          1.95              0.97       1.17               0.00              0.00
Noord  Basis              Spraak             4          4.74              1.19       2.35              91.00              0.00
Noord  Plus               SMS                4          3.00              0.75       0.97               0.00              0.00
Noord  Plus               Spraak             3          8.12              2.71       3.80             149.00              0.00
Noord  Prepaid            Data               2          2.16              1.08       1.09               0.00        


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Stap 3 - Materialiseer benoemde drill-subkubussen met TYPES

Een OLAP-kubus slaat de roll-ups die analisten het vaakst raadplegen vooraf op. De `TYPES`-instructie doet precies dat: elke term vraagt `PROC SUMMARY` om één subkubus af te leveren. We vragen het eindtotaal `()`, de `region`-marginaal, en de tweeweg-subkubussen `region*plan_tier` en `call_type*plan_tier` op - de drillpaden die een omzetdashboard zou tonen. SAS markeert elke subkubus met een `_TYPE_`-code (een bitmasker over de `CLASS`-lijst), zodat één outputdataset elk niveau van de hiërarchie bevat.

In [3]:
PROCEDURE summary GEGEVENS=work.cdr_billing;
    KLASSE region plan_tier call_type;
    VARIABELE revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    UITVOER out=work.cube_hier
           sum(revenue)=rev_total;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=work.cube_hier noobs label;
    label _type_='Subkubustype' region='Regio' plan_tier='Abonnementsniveau'
          call_type='Gesprekstype' _freq_='Aantal Records' rev_total='Totale Omzet';
    TITEL "Subkubus-optellingen: eindtotaal, regio, regio*niveau, gesprekstype*niveau";
    VARIABELE _type_ region plan_tier call_type _freq_ rev_total;
    OPMAAK rev_total comma10.2;
UITVOEREN;


                       Subkubus-optellingen: eindtotaal, regio, regio*niveau, gesprekstype*niveau                       

Subkubustype  Regio  Abonnementsniveau  Gesprekstype  Aantal Records  Totale Omzet
           0                                                     100        222.78
           3         Basis              Data                       8         38.06
           3         Basis              SMS                        8          8.03
           3         Basis              Spraak                    20         42.33
           3         Plus               Data                       3         17.46
           3         Plus               SMS                       13         11.75
           3         Plus               Spraak                    18         59.06
           3         Prepaid            Data                       7         14.50
           3         Prepaid            SMS                        7          6.82
           3         Prepaid            Spraak  


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Stap 4 - Gewogen projectie, een regionale drill en lekkage-triage

Drie analyses die een omzetbewakingsteam daadwerkelijk uitvoert tegen de kubus:

- **Gewogen projectie.** Door `WEIGHT=subscriber_wt` toe te voegen aan een `region*plan_tier`-samenvatting rapporteert deze omzet geschaald naar de volledige abonneebasis die de steekproef vertegenwoordigt, in plaats van de ruwe gesteekproefde rijen.
- **Drill.** Het filteren van de NWAY-kubus tot één regio breidt één tak van de hiërarchie uit - hier Zuid - naar het niveau-per-dienst detail.
- **Lekkage-triage.** Het sorteren van de cellen op gemiddelde omzet per record laat de cellen met de hoogste opbrengst naar voren komen; dunbezette, hoogopbrengende cellen zijn precies waar een audit op let voor verkeerd getarifeerde of lekkende omzet.

In [4]:
/* Gewogen omzet geprojecteerd naar de abonneebasis */
PROCEDURE summary GEGEVENS=work.cdr_billing NWAY;
    KLASSE region plan_tier;
    VARIABELE revenue;
    GEWICHT subscriber_wt;
    UITVOER out=work.cube_wt(VERWIJDEREN=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=work.cube_wt noobs label;
    label region='Regio' plan_tier='Abonnementsniveau'
          rev_weighted='Gewogen Omzet' records='Aantal Records';
    TITEL "Gewogen omzet per regio * abonnementsniveau (geprojecteerd op abonneebasis)";
    OPMAAK rev_weighted comma10.2;
UITVOEREN;

/* Inzoomen op de Zuid-tak van de kubus */
PROCEDURE AFDRUKKEN GEGEVENS=work.cube_nway noobs label;
    WAAR region = "Zuid";
    label plan_tier='Abonnementsniveau' call_type='Gesprekstype'
          rev_total='Totale Omzet' rev_mean='Gemiddelde Omzet';
    TITEL "Inzoomen op Zuid: omzetcellen per niveau en gesprekstype";
    VARIABELE plan_tier call_type _freq_ rev_total rev_mean;
    OPMAAK rev_total rev_mean comma10.2;
UITVOEREN;

/* Rangschik cellen op opbrengst per record voor lekkage-triage */
PROCEDURE SORTEREN GEGEVENS=work.cube_nway out=work.cube_ranked;
    VOLGENS AFLOPEND rev_mean;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=work.cube_ranked(obs=6) noobs label;
    label region='Regio' plan_tier='Abonnementsniveau' call_type='Gesprekstype'
          rev_mean='Gemiddelde Omzet' rev_max='Max Omzet';
    TITEL "Cellen met hoogste gemiddelde omzet (opbrengst per record)";
    VARIABELE region plan_tier call_type _freq_ rev_mean rev_max;
    OPMAAK rev_mean rev_max comma10.2;
UITVOEREN;


                      Gewogen omzet per regio * abonnementsniveau (geprojecteerd op abonneebasis)                       

Regio  Abonnementsniveau  Gewogen Omzet  Aantal Records
Noord  Basis                      18.30               7
Noord  Plus                       22.42               7
Noord  Prepaid                    20.56               9
Oost   Basis                      50.85              15
Oost   Plus                       59.59              12
Oost   Prepaid                    29.77              11
Zuid   Basis                      58.63              14
Zuid   Plus                       56.29              15
Zuid   Prepaid                    27.77              10

                                Inzoomen op Zuid: omzetcellen per niveau en gesprekstype                                

Abonnementsniveau  Gesprekstype  _FREQ_  Totale Omzet  Gemiddelde Omzet
Basis              Data               3         12.02              4.01
Basis              SMS                2          2.


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Interpretatie van de resultaten

De samenvattingskubus zet 100 ruwe abonnee-dagrijen om in een compacte verzameling vooraf geaggregeerde cellen die drill-downvragen direct beantwoorden, zonder de feitentabel opnieuw te doorzoeken:

- **Waar de omzet zit.** De `region`-marginaal (`_TYPE_=4`) toont dat Zuid $97,44 en Oost $86,94 verrekenden - samen 83% van de $222,78 van de maand - terwijl Noord $38,40 bijdroeg. Binnen de subkubus `call_type*plan_tier` (`_TYPE_=3`) is Plus-niveau Spraak de rijkste afzonderlijke cel met $59,06 over 18 records, met Basis-niveau Spraak als tweede op $42,33.
- **Gewogen projectie.** Het toepassen van het steekproefgewicht verschuift de rangschikking richting abonnementen waarvan de abonnees zwaarder wegen: Oost-Plus ($59,59) en Zuid-Basis ($58,63) voeren de geprojecteerde `region*plan_tier`-omzet aan, een ander beeld dan de ongewogen regiototalen en een herinnering om geprojecteerde in plaats van gesteekproefde omzet te rapporteren bij het inschatten van blootstelling.
- **Opbrengst per record en lekkage-triage.** Het rangschikken van NWAY-cellen op `rev_mean` zet datagebruikcellen bovenaan - Noord-Basis-Data ($7,87/record) en Zuid-Plus-Data ($5,96/record) - wat bevestigt dat intensief datagebruik de hoogste opbrengst per record oplevert. De dunbezette koplopers (één of twee records) zijn precies de cellen waarvoor een omzetbewakingsanalist de onderliggende call-detail records zou opvragen, om te bevestigen dat de hoge kosten correct getarifeerd zijn en geen fout betreffen.

Voor een omzetbewakingsteam is deze kubus de basis voor variantiedashboards: vergelijk verrekende omzet met de verwachte omzet volgens het tarievenblad per (regio, niveau, gesprekstype)-cel, en de cellen waarvan het gemiddelde of gewogen totaal het meest afwijkt, worden de lekkagegevallen die het waard zijn om te auditeren. Omdat de hele structuur een gewone SAS-dataset is, kan de kubus van de volgende maand met dezelfde Base SAS-tools worden samengevoegd (union), vergeleken (verschil), of gejoind met een tarievenblad.